# Dog, Cat & Panda Image Classifier

**AMV10 Visual Analytics — Group 14**

This notebook trains a ResNet-50 classifier on the Dog/Cat/Panda dataset
and exports all artefacts needed by the VA dashboard:
- `export.pkl` — fastai learner for LIME explanations
- `va_export/predictions.csv` — predictions for all images
- Model weights for retraining

Dataset: [Animal Image Dataset (Dog, Cat, Panda)](https://www.kaggle.com/datasets/ashishsaxena2209/animal-image-datasetdog-cat-and-panda)

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from fastai import *
from fastai.vision import *
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import json
from pathlib import Path
from PIL import Image as PILImage


In [ ]:
def seed_everything(seed):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

SEED = 42
seed_everything(SEED)
print('cudnn enabled:', torch.backends.cudnn.enabled)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Data Loading

The dataset has 3 folders: `dogs/`, `cats/`, `panda/`, each containing ~1000 images.
We build a DataFrame with image paths and labels (0=Cat, 1=Dog, 2=Panda).

In [ ]:
DATA_DIR = os.path.join('.', 'data')

CLASS_MAP = {'cats': 0, 'dogs': 1, 'panda': 2}
CLASS_NAMES = {0: 'Cat', 1: 'Dog', 2: 'Panda'}

rows = []
for folder, label in CLASS_MAP.items():
    folder_path = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(folder_path):
        print(f'WARNING: {folder_path} not found')
        continue
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            rows.append({
                'image_id': os.path.splitext(fname)[0],
                'path': os.path.join(folder_path, fname),
                'label': label,
                'class_name': CLASS_NAMES[label],
            })

df = pd.DataFrame(rows)
print(f'Total images: {len(df)}')
print(df['class_name'].value_counts())


In [ ]:
if df.empty:
    print('No data loaded, so class distribution cannot be plotted yet.')
else:
    ax = df['class_name'].value_counts().plot.bar(figsize=(6, 3), color=['#e74c3c', '#3498db', '#2ecc71'])
    plt.title('Class Distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()


In [ ]:
# Show sample images
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, cls in enumerate(['cats', 'dogs', 'panda']):
    sample = df[df['class_name'] == CLASS_NAMES[CLASS_MAP[cls]]].iloc[0]
    from PIL import Image as PILImage
    img = PILImage.open(sample['path'])
    axes[i].imshow(img)
    axes[i].set_title(sample['class_name'])
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## 2. Create fastai DataBunch

We use fastai's ImageDataBunch with standard augmentations.
This is a **classification** task (not regression like DR grading).

In [ ]:
bs = 64
sz = 224

tfms = get_transforms(do_flip=True, flip_vert=False, max_rotate=15,
                       max_warp=0, max_zoom=1.1, max_lighting=0.2)

data = (ImageList.from_df(df=df, path='', cols='path')
        .split_by_rand_pct(0.2, seed=SEED)
        .label_from_df(cols='class_name')
        .transform(tfms, size=sz)
        .databunch(bs=bs)
        .normalize(imagenet_stats))

print(f'Train: {len(data.train_ds)}, Valid: {len(data.valid_ds)}')
print(f'Classes: {data.classes}')


In [ ]:
data.show_batch(rows=3, figsize=(9, 6))

## 3. Train ResNet-50

Transfer learning from ImageNet. This is a standard 3-class classification
with cross-entropy loss (fastai default for categorical labels).

In [ ]:
learn = cnn_learner(data, base_arch=models.resnet50, metrics=[accuracy])


In [ ]:
learn.to_fp16()
learn.lr_find()
learn.recorder.plot(suggestion=True)

In [ ]:
learn.fit_one_cycle(4, max_lr=1e-2)

In [ ]:
learn.recorder.plot_losses()
learn.recorder.plot_metrics()

In [ ]:
learn.unfreeze()
learn.lr_find()
learn.recorder.plot(suggestion=True)

In [ ]:
learn.fit_one_cycle(6, max_lr=slice(1e-6, 1e-3))

In [ ]:
learn.recorder.plot_losses()
learn.recorder.plot_metrics()

In [ ]:
# Save model
learn.export()
learn.save('stage-2')
print('Saved export.pkl and models/stage-2.pth')

## 4. Evaluate

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(8, 8), dpi=60)

In [ ]:
interp.plot_top_losses(9, figsize=(12, 12))

## 5. Export artefacts for VA dashboard

We need:
1. `predictions.csv` — image_id, true_class, pred_class, confidence, class_confidences
2. The model is used live by the dashboard (no separate embeddings file needed — hidden layers extracted on the fly)

In [ ]:
EXPORT_DIR = os.path.join('.', 'va_export')
os.makedirs(EXPORT_DIR, exist_ok=True)

# Get predictions for ALL images (train + valid)
all_image_list = ImageList.from_df(df=df, path=DATA_DIR, cols='path')

# Add as test set and predict
learn.data.add_test(all_image_list)
preds, _ = learn.get_preds(ds_type=DatasetType.Test)

# preds is (N, 3) softmax probabilities
probs = preds.numpy()
pred_classes = probs.argmax(axis=1)

# fastai class order should match folder-name labels used in training
fastai_classes = data.classes  # e.g. ['cats', 'dogs', 'panda']
print(f'fastai class order: {fastai_classes}')

# Build class name -> numeric label mapping using folder names
name_to_label = {name: idx for idx, name in IDX_TO_CLASS.items()}
fastai_idx_to_label = [name_to_label[c] for c in fastai_classes]
print(f'fastai index -> our label: {fastai_idx_to_label}')


In [ ]:
# Build predictions DataFrame
predictions = []
for i in range(len(df)):
    row = df.iloc[i]
    prob_vec = probs[i]  # probabilities in fastai class order
    pred_idx = pred_classes[i]
    pred_label = fastai_idx_to_label[pred_idx]

    # Reorder probabilities to our fixed numeric-label order: 0=cats, 1=dogs, 2=panda
    ordered_probs = [float(prob_vec[fastai_classes.index(IDX_TO_CLASS[j])]) for j in range(len(IDX_TO_CLASS))]

    predictions.append({
        'image_id': row['image_id'],
        'path': row['path'],
        'folder_name': row['folder_name'],
        'true_class': int(row['label']),
        'true_class_name': row['class_name'],
        'pred_class': int(pred_label),
        'pred_class_name': IDX_TO_CLASS[int(pred_label)],
        'confidence': round(float(prob_vec.max()), 4),
        'class_confidences': json.dumps([round(p, 4) for p in ordered_probs]),
    })

pred_df = pd.DataFrame(predictions)
print(f'Predictions shape: {pred_df.shape}')
print(f'Accuracy: {(pred_df["true_class"] == pred_df["pred_class"]).mean():.1%}')
pred_df.head()


In [ ]:
# Save predictions
pred_df.to_csv(os.path.join(EXPORT_DIR, 'predictions.csv'), index=False)
print(f'Saved: {EXPORT_DIR}/predictions.csv ({len(pred_df)} rows)')

# Save class mapping
class_info = {
    'class_names': IDX_TO_CLASS,              # numeric label -> folder name
    'label_to_class': CLASS_MAP,              # folder name -> numeric label
    'fastai_classes': fastai_classes,
    'fastai_idx_to_label': fastai_idx_to_label,
    'num_classes': len(IDX_TO_CLASS),
}
with open(os.path.join(EXPORT_DIR, 'class_info.json'), 'w') as f:
    json.dump(class_info, f, indent=2)
print(f'Saved: {EXPORT_DIR}/class_info.json')


In [ ]:
print('\n=== Export Summary ===')
print(f'export.pkl          : model for LIME + dashboard')
print(f'va_export/predictions.csv : {len(pred_df)} predictions')
print(f'va_export/class_info.json : class mapping')
print(f'\nAccuracy: {(pred_df["true_class"] == pred_df["pred_class"]).mean():.1%}')
print(f'\nTo run the dashboard:')
print(f'  1. Copy export.pkl to the dashboard folder')
print(f'  2. Copy va_export/ to the dashboard folder')
print(f'  3. Make sure the images folder is accessible')
print(f'  4. python app_DCP.py')